PySpark Interview Questions 

Day 1 — Scenario-Based Scenario

Question :
You have an employee attendance DataFrame:

attendance_df

| Column Name | Type |
| ------------- | ---------------------------- |
| `emp_id` | string |
| `login_time` | string (yyyy-MM-dd HH:mm:ss) |
| `logout_time` | string (yyyy-MM-dd HH:mm:ss) |
| `location` | string |

Scenario Requirements:

Your manager wants an Employee Daily Working Hours Report:

1. Convert `login_time` and `logout_time` to proper timestamp.
2. Calculate working_hours = difference between logout and login in hours (decimal format).
3. Extract date from login_time.
4. Generate daily summary:

* total_hours_worked per employee per day
* first_login_time
* last_logout_time
5. Output columns:
`emp_id, date, total_hours_worked, first_login_time, last_logout_time`
6. Sort by `emp_id`, `date`.

Provide a single combined PySpark solution.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

data = [
    ('EMP1001', '2026-05-19 08:45:12', '2026-05-19 17:32:48', 'Hyderabad'),
    ('EMP1002', '2026-05-19 09:10:05', '2026-05-19 18:05:27', 'Bengaluru'),
    ('EMP1003', '2026-05-19 07:58:44', '2026-05-19 16:49:11', 'Chennai'),
    ('EMP1004', '2026-05-19 10:02:19', '2026-05-19 19:15:36', 'Mumbai'),
    ('EMP1005', '2026-05-19 08:20:30', '2026-05-19 17:00:09', 'Pune'),
    ('EMP1006', '2026-05-19 09:35:50', '2026-05-19 18:22:14', 'Visakhapatnam'),
    ('EMP1001', '2026-02-19 08:45:12', '2026-03-19 17:32:48', 'Hyderabad'),
    ('EMP1002', '2026-05-19 09:10:05', '2026-06-19 18:05:27', 'Bengaluru'),
]

schema=StructType([
    StructField('emp_id',StringType(),False ),
    StructField('login_time',StringType(),True),
    StructField('logout_time',StringType(),True),
    StructField('location',StringType(), True)
])

attendance_df=spark.createDataFrame(data,schema)
#PySpark cannot directly subtract string timestamps, so we first convert them into Unix timestamps,
#This converts the datetime string into seconds from Jan 1, 1970 (Epoch time).
#Example:2026-05-19 08:45:12'→ 1779176712, 1 hour = 3600 seconds
attendance_df = attendance_df.withColumn(
    "date",
    to_date("login_time"))
windowSpec=Window.partitionBy('emp_id','date').orderBy('emp_id','date')
attendance_df=attendance_df.withColumn('login_time', to_timestamp(col('login_time')))
attendance_df=attendance_df.withColumn('login_time', to_timestamp(col('login_time'))).withColumn('logout_time', to_timestamp(col('logout_time'))).withColumn('working_hours', (((unix_timestamp(col('logout_time')) - unix_timestamp(col('login_time'))) / 3600)).cast('decimal(10,2)')).withColumn('first_login_time',to_date(min('login_time').over(windowSpec))).withColumn('last_logout_time',to_date(max('logout_time').over(windowSpec)))
attendance_df=attendance_df.withColumn('total_hours_worked',sum(col('working_hours')).over(windowSpec)).drop('login_time','logout_time','working_hours','location').show(truncate=False)


PySpark Interview Questions 

Day 2 — Scenario-Based Question

Question :

You are working in a banking system.
You have a loan repayment dataset:

loan_payments_df :

| Column Name | Type |
| ---------------- | ------------------- |
| `customer_id` | string |
| `loan_id` | string |
| `payment_amount` | double |
| `payment_date` | string (yyyy-MM-dd) |
| `loan_amount` | double |

Scenario Requirements :

Build a Loan Repayment Tracking Report with the following requirements:

1. Convert `payment_date` to DateType.
2. For each loan:

* Calculate total_paid_amount
* Calculate remaining_balance
* Calculate repayment_percentage
3. Identify loan status:

* repayment_percentage >= 100 → `"PAID_OFF"`
* repayment_percentage between 50 and 99 → `"IN_PROGRESS"`
* repayment_percentage < 50 → `"LOW_RECOVERY"`
4. Find latest payment date per loan.
5. Output columns:
`customer_id, loan_id, loan_amount, total_paid_amount, remaining_balance, repayment_percentage, latest_payment_date, loan_status`
6. Sort by `repayment_percentage` descending.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

data = [
    # PAID_OFF (>=100%)
    ("CUST001", "LOAN1001", 50000.00, "2026-01-15", 100000.00),
    ("CUST001", "LOAN1001", 30000.00, "2026-02-15", 100000.00),
    ("CUST001", "LOAN1001", 25000.00, "2026-03-15", 100000.00),

    ("CUST002", "LOAN1002", 80000.00, "2026-01-10", 150000.00),
    ("CUST002", "LOAN1002", 70000.00, "2026-02-10", 150000.00),
    ("CUST002", "LOAN1002", 10000.00, "2026-03-10", 150000.00),

    # IN_PROGRESS (50% - 99%)
    ("CUST003", "LOAN1003", 20000.00, "2026-01-12", 80000.00),
    ("CUST003", "LOAN1003", 25000.00, "2026-02-12", 80000.00),

    ("CUST004", "LOAN1004", 60000.00, "2026-01-20", 120000.00),
    ("CUST004", "LOAN1004", 20000.00, "2026-02-20", 120000.00),

    ("CUST005", "LOAN1005", 40000.00, "2026-01-18", 90000.00),
    ("CUST005", "LOAN1005", 15000.00, "2026-02-18", 90000.00),

    # LOW_RECOVERY (<50%)
    ("CUST006", "LOAN1006", 10000.00, "2026-01-25", 100000.00),
    ("CUST006", "LOAN1006", 12000.00, "2026-02-25", 100000.00),

    ("CUST007", "LOAN1007", 15000.00, "2026-01-05", 80000.00),

    ("CUST008", "LOAN1008", 20000.00, "2026-01-08", 150000.00),
    ("CUST008", "LOAN1008", 10000.00, "2026-02-08", 150000.00)

]


schema=StructType([
    StructField('customer_id',StringType(),False),
    StructField('loan_id',StringType(),True),
    StructField('payment_amount',DoubleType(),True),
    StructField('payment_date',StringType(),True),
    StructField('loan_amount',DoubleType(),True)
])
windowspec=Window.partitionBy(col("loan_id"))
repayment_tracking_df= spark.createDataFrame(data,schema)
#1
repayment_tracking_df=repayment_tracking_df.withColumn("payment_date", to_date(col("payment_date"), "yyyy-MM-dd"))
#2
#For each loan:
#Calculate total_paid_amount
#Calculate remaining_balance
#Calculate repayment_percentage
repayment_tracking_df=repayment_tracking_df.withColumn("total_paid_amount", sum(col("payment_amount")).over(windowspec))
repayment_tracking_df=repayment_tracking_df.withColumn("remaining_balance", col("loan_amount")-col("total_paid_amount"))
repayment_tracking_df=repayment_tracking_df.withColumn("repayment_percentage", round((col("total_paid_amount")/col("loan_amount"))*100, 2))
#3 Identify loan status:
repayment_tracking_df=repayment_tracking_df.withColumn("loan_status", when(col("repayment_percentage")>=100, "PAID_OFF")\
                                                       .when((col("repayment_percentage")>=50) & (col("repayment_percentage")<100),"IN_PROGRESS")\
                                                       .otherwise("LOW_RECOVERY"))
#latest_payment_date
repayment_tracking_df=repayment_tracking_df.withColumn("latest_payment_date", max(col("payment_date")).over(windowspec))
repayment_tracking_df=repayment_tracking_df.drop("payment_amount","payment_date").orderBy(col("repayment_percentage").desc())
repayment_tracking_df=repayment_tracking_df.dropDuplicates()
repayment_tracking_df.show()


PySpark Interview Questions 

Day 3 — Scenario-Based Question

Question :
You are working for a streaming platform like Netflix.
You have a viewing dataset:

watch_history_df

| Column Name | Type |
| ---------------- | ------------------- |
| `user_id` | string |
| `content_id` | string |
| `watch_time_min` | integer |
| `watch_date` | string (yyyy-MM-dd) |
| `genre` | string |

Scenario Requirements :

Build a User Watch Analytics Report with the following requirements:

1. Convert `watch_date` to DateType.
2. For each user and month:

* total_watch_time
* total_content_watched
* favorite_genre (genre watched most frequently)
3. Add a binge_watcher flag:

* total_watch_time > 3000 minutes/month → `"YES"`
* otherwise `"NO"`
4. Output columns:
`user_id, year, month, total_watch_time, total_content_watched, favorite_genre, binge_watcher`
5. Sort by `user_id`, `year`, `month`.


PySpark Interview Questions 

Day 4 — Scenario-Based Question

Question :
You are working in a logistics company.
You have a shipment tracking dataset:

shipment_df :

| Column Name | Type |
| ----------------- | ---------------------------- |
| `shipment_id` | string |
| `vehicle_id` | string |
| `shipment_status` | string |
| `status_time` | string (yyyy-MM-dd HH:mm:ss) |
| `delivery_city` | string |

Scenario Requirements :

Build a Shipment Delivery Performance Report with the following requirements:

1. Convert `status_time` to TimestampType.
2. For each shipment:

* Find shipment start time (`first status_time`)
* Find shipment end time (`last DELIVERED status_time`)
3. Calculate:

* total_delivery_time_hours
4. Identify delivery performance:

* <= 24 hrs → `"ON_TIME"`
* > 24 hrs and <= 48 hrs → `"DELAYED"`
* > 48 hrs → `"CRITICAL_DELAY"`
5. Output columns:
`shipment_id, vehicle_id, delivery_city, start_time, delivered_time, total_delivery_time_hours, delivery_status`
6. Sort by `total_delivery_time_hours` descending.

PySpark Interview Questions 

Day 5 — Scenario-Based Question

Question :
You are working on an e-commerce recommendation system.
You have a customer orders dataset:

orders_df

| Column Name | Type |
| ------------- | ------------------- |
| `customer_id` | string |
| `order_id` | string |
| `product_id` | string |
| `order_date` | string (yyyy-MM-dd) |
| `amount` | double |

Scenario Requirements :

Build a Customer Purchase Frequency Report with the following requirements:

1. Convert `order_date` to DateType.
2. For each customer:

* Calculate days between consecutive orders.
* Find average_days_between_orders.
3. Identify customer category:

* avg_days_between_orders <= 7 → `"FREQUENT_BUYER"`
* avg_days_between_orders <= 30 → `"REGULAR_BUYER"`
* otherwise `"OCCASIONAL_BUYER"`
4. Also calculate:

* total_orders
* total_spent
5. Output columns:
`customer_id, total_orders, total_spent, average_days_between_orders, customer_category`
6. Sort by `total_spent` descending.

Provide a single combined PySpark solution.

PySpark Interview Scenario-Based Question

Question 6 :

You have a transaction dataset:

transactions_df :

| Column Name | Type |
| ------------------ | ------------------- |
| `account_id` | string |
| `transaction_id` | string |
| `transaction_date` | string (yyyy-MM-dd) |
| `amount` | double |

Scenario Requirements :

Build an Account Balance Trend Report with the following logic:

1. Convert `transaction_date` to DateType.
2. For each account:

* Calculate daily total transaction amount.
* Calculate running account balance ordered by date.
3. Identify days where:

* running_balance < 0 → `"NEGATIVE_BALANCE"`
* otherwise `"HEALTHY"`
4. Add previous day balance using window function.
5. Output columns:
`account_id, transaction_date, daily_total, previous_balance, running_balance, account_status`
6. Sort by `account_id`, `transaction_date`.

Provide a single combined PySpark solution.

PySpark Interview Questions — Day 7 

Scenario-Based Question :
You have a large dataset of user transactions:

transactions_df

| Column Name | Type |
| ------------------ | ---------------------------- |
| `user_id` | string |
| `transaction_id` | string |
| `amount` | double |
| `transaction_time` | string (yyyy-MM-dd HH:mm:ss) |

Scenario Requirements :
You need to build a User Spending Behavior Report:

1. Convert `transaction_time` to timestamp.
2. Extract `date` from transaction_time.
3. For each user per day:

* total_spent
* number_of_transactions
4. Identify high spending days:

* If `total_spent > 5000` → mark as `"HIGH_SPENDER"`
* Else → `"NORMAL"`
5. Additionally, calculate cumulative_spent per user (running total ordered by date).
6. Output columns:
`user_id, date, total_spent, number_of_transactions, spending_flag, cumulative_spent`
7. Sort by `user_id`, `date`.

Provide a single combined PySpark solution.

PySpark Interview Questions 

Day 8 — Scenario-Based Question

Question :
You have a DataFrame containing IoT sensor readings:

sensor_df

| Column Name | Type |
| -------------- | ---------------------------- |
| `device_id` | string |
| `reading_time` | string (yyyy-MM-dd HH:mm:ss) |
| `temperature` | double |
| `humidity` | double |
| `location` | string |

Scenario Requirements :
You must build a Daily Sensor Health Monitoring Report:

1. Convert `reading_time` to timestamp.
2. Extract `date` from `reading_time`.
3. For each device per day:

* avg_temperature
* max_temperature
* min_temperature
* avg_humidity
* reading_count
4. Add a new column health_status using rules:

* If `avg_temperature > 80` OR `avg_humidity > 70` → `"CRITICAL"`
* If `avg_temperature between 60 and 80` → `"WARNING"`
* Else → `"NORMAL"`
5. Output columns:
`device_id, date, avg_temperature, max_temperature, min_temperature, avg_humidity, reading_count, health_status`
6. Sort by `device_id`, `date`.

Provide a single combined PySpark solution.


PySpark Interview Questions
Day 9 — Scenario-Based Question

Question :
You have two DataFrames:

products_df

| Column Name | Type |
| -------------- | ------ |
| `product_id` | string |
| `product_name` | string |
| `category` | string |

sales_df

| Column Name | Type |
| ------------ | ------------------- |
| `sale_id` | string |
| `product_id` | string |
| `quantity` | integer |
| `price` | double |
| `sale_date` | string (yyyy-MM-dd) |

Scenario Requirements :
You must build a Category-Wise Monthly Revenue Report:

1. Convert `sale_date` to DateType.
2. Calculate revenue = quantity * price.
3. Extract year and month.
4. Join `sales_df` with `products_df` to get category.
5. For each category per month:

* total_revenue
* total_quantity
* average_price_per_unit (total_revenue / total_quantity)
6. Output columns:
`category, year, month, total_revenue, total_quantity, average_price_per_unit`
7. Sort by `category`, `year`, `month`.

Provide a single combined PySpark solution.

PySpark Interview Questions 

Day 10 — Scenario-Based Question

Question :
You have a web activity DataFrame:

web_logs_df

| Column Name | Type |
| ------------ | ---------------------------- |
| `user_id` | string |
| `event_time` | string (yyyy-MM-dd HH:mm:ss) |
| `page` | string |
| `session_id` | string |

Scenario Requirements:

You need to create a User Session Activity Summary Report with the following requirements:

1. Convert `event_time` to timestamp.
2. Extract `event_date` from `event_time`.
3. For each user per session:

* Count total page_visits
* Identify first_event_time
* Identify last_event_time
* Calculate session_duration_minutes (difference between last and first event)
4. Output columns:
`user_id, session_id, event_date, page_visits, first_event_time, last_event_time, session_duration_minutes`
5. Sort output by `user_id`, `event_date`, `session_id`.

Provide a single combined PySpark solution

PySpark Interview Questions
Day 11 — Scenario-Based Question

Question
You have an orders DataFrame:
orders_df

| Column Name | Type |
| -------------- | ------------------- |
| `order_id` | string |
| `customer_id` | string |
| `order_amount` | double |
| `order_date` | string (yyyy-MM-dd) |

Your task is to generate a Customer Order Trend Report with the following requirements:

1. Convert `order_date` to proper DateType.
2. Determine **previous order amount** for each customer (use window function).
3. Calculate:

* `difference_from_prev_order`
* `percent_change_from_prev_order`
4. If no previous order exists, set both values to 0.
5. Return:
`customer_id, order_id, order_amount, order_date, difference_from_prev_order, percent_change_from_prev_order`
6. Sort the final result by `customer_id`, `order_date`.

Provide one combined PySpark solution.

PySpark Interview Questions
Day 12 — Scenario-Based Question


Question :
You receive two DataFrames:

1. customer_df

| Column Name | Type |
| ----------- | ------ |
| `cust_id` | string |
| `name` | string |
| `country` | string |

2. transaction_df

| Column Name | Type |
| ----------------- | ---------------------------- |
| `trans_id` | string |
| `cust_id` | string |
| `amount` | double |
| `trans_timestamp` | string (yyyy-MM-dd HH:mm:ss) |

Scenario:

Your manager wants the following **Customer Transaction Insights Report**:

1. Convert `trans_timestamp` to proper timestamp.
2. Determine the **latest transaction amount** for each customer.
3. Join this information with `customer_df`.
4. If a customer has **no transactions**, show `latest_amount = 0`.
5. Output columns:
`cust_id`, `name`, `country`, `latest_amount`
6. Sort by `latest_amount` descending.

Provide one combined PySpark solution (no separate answers).


Day 13 – PySpark Scenario-Based Interview Question

Scenario:

You are working in an e-commerce analytics platform.
You receive a product catalog dataset and sales dataset.

1️⃣ products (2M records)

product_id (string)
product_name (string)
category (string)
price (double)

2️⃣ sales (900M records)

sale_id (string)
product_id (string)
quantity (integer)
sale_date (date)


Business Requirement:

1. Find total quantity sold per category per day.
2. The sales table is extremely large.
3. Join should be optimized.
4. Output should contain:

sale_date
category
total_quantity

How would you implement this in PySpark efficiently?

Day 14 – PySpark Scenario-Based Interview Question

Scenario:

You are working in a large telecom company.
You receive customer usage logs that contain daily data.

Dataset: usage_logs (1.2 billion records)

```id="ukdncu"
customer_id (string)
usage_mb (double)
call_minutes (double)
sms_count (int)
event_date (date)
circle (string) -- region
```

Business Requirement:

1. For each customer, compute a 30-day rolling average of:

* usage_mb
* call_minutes
* sms_count
2. Rolling window must be based on event_date.
3. Data is huge → must be optimized.
4. Final output should contain:

```id="itklb7"
customer_id
event_date
avg_usage_30d
avg_call_minutes_30d
avg_sms_30d
```

How would you implement this efficiently in PySpark?

Day 15 – PySpark Scenario-Based Interview Question

 Scenario:

You are working in a ride-sharing company (like Uber/Ola). You receive trip data daily.
Schema:

trip_id (string)
driver_id (string)
city (string)
trip_distance (double)
fare_amount (double)
trip_date (date)
```

Dataset size: 700M+ records

Business Requirement:

1. Find the top 5 drivers in each city based on total fare_amount.
2. The solution must scale efficiently for hundreds of millions of records.
3. Output should contain:

* city
* driver_id
* total_fare
* rank

How would you implement this using PySpark?